# OVD-Diagnose — cross-domain OVD failure benchmark

Runs frozen open-vocabulary detectors over a domain in Global / Oracle / Agnostic modes and
computes each (L, S, C) failure fingerprint. See `paper/DESIGN.md`.

**Headless-ready:** Save Version → Save & Run All (Commit). Accelerator = GPU, Internet = On.
Everything runs via `!python` (fresh process per cell — no kernel-state or autoreload needed).
Outputs land in `/kaggle/working/runs/diag/aerial/fingerprints.csv` (captured as notebook Output).

In [ ]:
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| GPUs', torch.cuda.device_count())

In [ ]:
# Clone / update repo
REPO = 'https://github.com/ShMazumder/YOLOBERT.git'
import os
if not os.path.isdir('/kaggle/working/YOLOBERT'):
    !git clone $REPO /kaggle/working/YOLOBERT
%cd /kaggle/working/YOLOBERT
!git pull -q || true

In [ ]:
# Deps (ultralytics -> YOLO-World + SAM; transformers -> OWLv2 + Grounding DINO)
!pip install -q ultralytics transformers pycocotools pyyaml
!python tools/ovd_diagnose.py   # metric self-test

## Aerial domain (LAE-80C) — download + recursive flatten, verified against annotation count

In [ ]:
import os, zipfile, shutil, json
BASE = '/kaggle/working/YOLOBERT/data/aerial'
ANN_DIR, IMG_DIR = f'{BASE}/annotations', f'{BASE}/images'
os.makedirs(ANN_DIR, exist_ok=True); os.makedirs(IMG_DIR, exist_ok=True)
ann = f'{ANN_DIR}/instances_val.json'
HF = 'https://huggingface.co/datasets/jaychempan/LAE-1M/resolve/main/LAE-80C'

if not os.path.exists(ann):
    !wget -q --no-check-certificate "{HF}/LAE-80C-benchmark.json?download=true" -O {ann}
    !wget -q --no-check-certificate "{HF}/LAE-80C-benchmark_categories.json?download=true" -O {ANN_DIR}/categories.json
    !wget -q --no-check-certificate "{HF}/LAE-80C-benchmark.txt?download=true" -O {ANN_DIR}/classes.txt

def count_jpgs():
    return sum(1 for _, _, fs in os.walk(IMG_DIR) for f in fs if f.lower().endswith('.jpg'))

n_expected = len(json.load(open(ann))['images'])

# download + extract only if we don't already have enough image files anywhere under IMG_DIR
if count_jpgs() < n_expected:
    zp = '/kaggle/working/images.zip'
    if not os.path.exists(zp) or os.path.getsize(zp) < 5e8:
        !wget -q --no-check-certificate "{HF}/images.zip?download=true" -O {zp}
    with zipfile.ZipFile(zp) as z:
        z.extractall(IMG_DIR)
    if os.path.exists(zp):
        os.remove(zp)

# recursive flatten: move every .jpg (any depth) to IMG_DIR root
for root, _, files in os.walk(IMG_DIR):
    if root == IMG_DIR:
        continue
    for f in files:
        if f.lower().endswith('.jpg'):
            dst = os.path.join(IMG_DIR, f)
            if not os.path.exists(dst):
                shutil.move(os.path.join(root, f), dst)
# prune empty subdirs
for root, dirs, files in os.walk(IMG_DIR, topdown=False):
    if root != IMG_DIR and not os.listdir(root):
        os.rmdir(root)

root_jpgs = len([f for f in os.listdir(IMG_DIR) if f.lower().endswith('.jpg')])
print(f'images (root): {root_jpgs} | expected: {n_expected} | complete: {root_jpgs >= n_expected}')

## Run all models (headless)
SAM once (shared `L`), then each OVD model in Global+Oracle. Per-model try/except — one
failure won't abort the Commit. `--limit 200` for a quick pass; drop it for the full split.
Writes `fingerprints.csv` + per-model json to the out dir.

In [ ]:
!python tools/run_all.py \
  --ann  data/aerial/annotations/instances_val.json \
  --imgs data/aerial/images \
  --out  runs/diag/aerial \
  --device cuda:0 --limit 200 \
  --models "yoloworld:yolov8s-world.pt,owlv2:google/owlv2-base-patch16-ensemble,groundingdino:IDEA-Research/grounding-dino-tiny" \
  --sam_weights mobile_sam.pt

In [ ]:
# Fingerprint table
import pandas as pd
df = pd.read_csv('runs/diag/aerial/fingerprints.csv')
print(df.to_string(index=False))

**Anchor (validates harness):** IoA-F1 per model should track Paper 1 — OWLv2 highest (~0.28),
YOLO-World ~0.03, Grounding DINO conservative. Read the fingerprint: high `S_norm` = semantic-confusion
bound; high `L` = localization bound; high `C_ece` = calibration bound.

Next: agriculture domain (same driver, new `--ann/--imgs`) → the cross-domain contrast that is the paper.